# Programming Assignment 1: A* Search

**Author:** John Louis D. Lagramada<br>

**Date:** February 13, 2026<br>

**Link:** [Colab Notebook](https://colab.research.google.com/drive/1hS0lzB4s-L_g_b1_n3vCzUznKua31Wtl?usp=sharing)

**A\* search** is a heuristic search algorithm that produces a *complete* and *optimal* solution given that the heuristic function $h(n)$ is admissible. Heuristic search chooses the state with the least total path cost, defined by the evaluation function $f(n)=g(n)+h(n)$. This is the sum of the current path cost and the heuristic cost estimate to reach the goal.

We followed the A\* search pseudocode provided in Lecture 3B: Heuristic Search. **No LLMs were used in writing this code. Please refer to the attached images for rough notes, and the references used in this assignment.**

**References:**

1. [Nilsson's Sequence Score](https://cs.stackexchange.com/questions/1904/nilssons-sequence-score-for-8-puzzle-problem-in-a-algorithm)

2. [Manhattan Distance](https://www.geeksforgeeks.org/data-science/manhattan-distance/)

3. [Dunder Methods in Python](https://www.geeksforgeeks.org/python/dunder-magic-methods-python/)

4. [Lecture 3B: Heuristic Search](https://www.dropbox.com/scl/fi/wmopmm0ufl7kgosghnmtm/Lecture-3B-AI201-Heuristic-Search.pdf?rlkey=tpabwuotgwa9zuu03baictto5&e=1&dl=0)

5. [Personal Notes](https://drive.google.com/file/d/10rlyvn5pADlJ3voIdkUX39EAWtZTJj1L/view?usp=sharing)
<br>

---

<br>

## States
A state in A\* search is represented as a *node*. In a node, we encapsulate the properties of a given state, in this case, we have:

- the current world state

- a pointer to a parent for finding solution path

- the path cost to get to state $g$

- the estimated path cost to get to goal $h$

- the total evaluation function $f = g+h$

- the Nilsson scores (P, S) if Nilsson heuristic is used

In [ ]:
class Node:
  def __init__(self, state, parent, g, h, nilsson_scores=None):
    self.state = state
    self.parent = parent
    self.g = g
    self.h = h
    self.f = g + h
    self.nilsson_scores = nilsson_scores

  # Comparing nodes is based on the evaluation function
  def __lt__(self, other):
    return self.f < other.f

  # Node equality is based on the state value
  def __eq__(self, other):
    return self.state == other.state

  # Verbosity in printing states and function values
  def __repr__(self):
    repr = ""
    for row in self.state:
      repr += str(row) + "\n"
    return f"{repr}\nf({self.f}) = g({self.g}) + h({self.h})\n"

## Heuristic Functions

We used three (3) heuristic functions:

1. the number of misplaced tiles

2. the manhattan distance

3. the Nilsson's sequence score

A good heuristic function never overestimates the actual cost to reach the goal. Why is this important? For example, there are two paths that lead to a goal. Path A *actually* costs less while Path B *actually* costs more. If we overestimate the cost of Path A such that $f_A > f_B$, A* will find Path B without traversing to Path A, even if Path A is the optimal path, breaking **optimality**.

Since we can always underestimate, what makes a good estimate? A good heuristic function (b) produces the closest approximation to the real path cost to reach the goal. By choosing a better heuristic this way, we can reduce the search cost within the bounds of completeness and optimality.

`heuristic_misplaced` calculates the number of misplaced tiles in the state relative to the goal state.

In [ ]:
def heuristic_misplaced(state, goal):
  misplaced = 0
  for i in range(3):
    for j in range(3):
      if state[i][j] != goal[i][j]:
        misplaced += 1

  return misplaced

`heuristic_manhattan` calculates the Manhattan distance (taxicab distance) of a misplaced tile to the goal tile position. It is defined as the 1-norm, from a class of p-norms.

$$
||x||_1=\sum_{j=1}^n|x_j|=|x_1 - x_2|+|y_1 - y_2|
$$

In [ ]:
# Find the current tile in the goal
def find_in_goal(value, goal):
  size = len(goal)
  for i in range(size):
    for j in range(size):
      if goal[i][j] == value:
        return i, j

# Calculate Manhattan distancce
def heuristic_manhattan(state, goal):
  distance = 0
  size = len(goal)
  for i in range(size):
    for j in range(size):
      index = find_in_goal(state[i][j], goal) # find tile
      distance += abs(i - index[0]) + abs(j - index[1])
  return distance

`heuristic_nilsson` calculates the Nilsson's sequence score as the sum of the Manhattan distance and $3S(x)$.

$$
N(x) = P(x) + 3S(x)
$$

$S(x)$ is simply twice the clockwise adjacent pairs in the state that are not in the clockwise adjacent pairs of the goal plus 1 if the center of the state is not equal to the center of the goal.
$$
S(x) = \begin{cases}
0, & \text{if } s_{ij} = g_{ij} \\
1, & \text{if } s_{ij}\neq g_{ij}\end{cases} +2(\text{CW adjacent S pairs not in G pairs})
$$

In [ ]:
# Get clockwise adjacent pairs of a 3x3 puzzle
def get_pairs(state):
  pairs = [[(0,0), (0,1)],
           [(0,1), (0,2)],
           [(0,2), (1,2)],
           [(1,2), (2,2)]]

  # Refer on Personal Notes: we observed that the first four (4) pairs
  # can be index as [(A, B), (C, D)] and the next four (4) pairs can
  # be indexed as its double inverse [(D, C), (A, B)].
  state_pairs = []
  for i in range(4):
    coords = pairs[i]
    state_pair = []
    inv_state_pair = []
    for coord in coords:
      state_pair.append(state[coord[0]][coord[1]]) # first pair
      inv_state_pair = [state[coord[1]][coord[0]]] + inv_state_pair # inverse pair
    state_pairs.append(state_pair)
    state_pairs.append(inv_state_pair)

  return state_pairs

# Get the Nilsson's sequence score
def heuristic_nilsson(state, goal, goal_pairs=None):
  # Get Manhattan distance
  P = heuristic_manhattan(state, goal)

  # Add 1 if the center tile is wrong
  S = 0
  if state[1][1] != goal[1][1]:
    S += 1

  # Get state and goal pairs
  state_pairs = get_pairs(state)
  if not goal_pairs:
    goal_pairs = get_pairs(goal)

  # Iterate if there are state pairs not in goal pairs
  for i in range(len(state_pairs)):
    if state_pairs[i] not in goal_pairs:
      S += 2

  # Return P & S
  return P, S

## Helper Functions

Helper functions are used to make the code more readable and decompose the logic for easier debugging.



`parse_input` is a function that obtains the start and goal states defined in the input text file.

In [ ]:
def parse_input(filename):
  with open(filename, 'r') as f:
    f.readline() # throw "start"

    # Get start state
    start = []
    for _ in range(3):
      line = f.readline()
      start.append([x for x in line.split()])

    f.readline() # throw "goal"

    # Get goal state
    goal = []
    for _ in range(3):
      line = f.readline()
      goal.append([x for x in line.split()])

  return start, goal

`lowest_cost` returns the node with the lowest cost among a list of nodes.

In [ ]:
def lowest_cost(nodes):
  cheapest = nodes[0]
  for node in nodes[1:]:
    if node < cheapest: # we overridden the __lt__ method for nodes
      cheapest = node
  return cheapest

`locate_blank` locates the `*` character in the current state since its position defines the only allowed moves.

In [ ]:
def locate_blank(state):
  for i in range(3):
    for j in range(3):
      if state[i][j] == '*':
        return i, j

`get_successors` is the successor function that generates the successors of a given state being expanded. It automatically calculates $f(n)$, $g(n)$, and $h(n)$ for the newly discovered nodes.

In [ ]:
def get_successors(node, heuristic, goal, goal_pairs=None):
  state = node.state
  blank_x, blank_y = locate_blank(state) # locate * coordinates
  successors = []

  # Helper function that creates a new instance of a node
  # if the state is a valid state. Given new positions for *
  # it swaps it with the value at that position.
  def add_node(new_blank_x, new_blank_y):
    state_copy = [row[:] for row in state]
    state_copy[blank_x][blank_y] = state_copy[new_blank_x][new_blank_y]
    state_copy[new_blank_x][new_blank_y] = '*'

    g = node.g + 1 # calculate g(n)

    # Pass goal_pairs to the heuristic function if it's nilsson
    if heuristic.__name__ == "heuristic_nilsson":
      P, S = heuristic(state_copy, goal, goal_pairs=goal_pairs)
      h = P + 3*S
      successors.append(Node(state_copy, node, g, h, (P, S)))
    else:
      h = heuristic(state_copy, goal) # calculate h(n)
      successors.append(Node(state_copy, node, g, h))

  # up
  if blank_x > 0:
    add_node(blank_x - 1, blank_y)

  # down
  if blank_x < 2:
    add_node(blank_x + 1, blank_y)

  # left
  if blank_y > 0:
    add_node(blank_x, blank_y - 1)

  # right
  if blank_y < 2:
    add_node(blank_x, blank_y + 1)

  return successors

`find_path` finds the path to the goal by traversing backwards from the tree generated by node expansion. Note that the tree is emergent because we used a node that can link to other nodes, the assignment criterion that we have to use lists is not broken as we will see later.

In [ ]:
def find_path(state):
  path = []
  counter = 0

  # Traverse backwards up to the start state
  while state.parent:
    path.append(state)
    state = state.parent
    counter += 1
  print(f"It took A* {counter} steps.")
  return path, counter

## A* Search

Finally, we write `a_star`, the function that performs the A* search. The algorithm is pretty straightforward once we have programmed the helper functions. We follow the pseudocode presented in Lecture 3B: Heuristic Search.

In [ ]:
def a_star(filename, heuristic, verbose=False):
  # Parse the initial state and the goal states
  start, goal = parse_input(filename)

  # Initialize OPEN and CLOSED lists
  OPEN = []
  CLOSED = []

  # If heuristic is Nilsson, get goal pairs
  # This will save compute because the goal state is
  # constant across runs, we want to compute it once.
  goal_pairs_for_nilsson = None
  if heuristic.__name__ == 'heuristic_nilsson':
    goal_pairs_for_nilsson = get_pairs(goal)

  # We initialize the start node.
  if heuristic.__name__ == 'heuristic_nilsson':
    P, S = heuristic(start, goal, goal_pairs=goal_pairs_for_nilsson)
    h = P + 3*S
    start_node = Node(start, None, 0, h, (P, S))
  else:
    h = heuristic(start, goal)
    start_node = Node(start, None, 0, h)
  OPEN.append(start_node)

  # We sofly initialize the goal node.
  goal_node = Node(goal, None, 0, 0)


  print(f"Using {heuristic.__name__}.")
  print(f"Initializing with state: \n{start_node}")
  print(f"Searching for state: \n{goal_node}")


  # Running the search
  i = 0
  while True:
    i += 1
    if verbose: print(f"Iteration {i}:")

    # If there are no more nodes to be expanded,
    # there are no solutions.
    if not OPEN:
      print("No solution!")
      return None

    # Get the state that has the lowest cost.
    n = OPEN.pop(OPEN.index(lowest_cost(OPEN)))
    CLOSED.append(n)

    # If heuristic is Nilsson, print f, g, h, P, and S
    if heuristic.__name__ == "heuristic_nilsson":
      print(f"State has f={n.f}, g={n.g}, h={n.h}, P={n.nilsson_scores[0]}, S={n.nilsson_scores[1]}.")
    else:
      print(f"State has f={n.f}, g={n.g}, h={n.h}.")

    if verbose: print(f"Operating on state:\n{n}")

    # If the goal is reached we traverse the path in reaching the goal.
    if n == goal_node:
      print(f"Goal reached after {i} iterations!")
      print(f"Goal reached after {len(OPEN) + len(CLOSED)} nodes discovered.")
      path, length = find_path(n)
      return (path, length, i, len(OPEN) + len(CLOSED))

    # Pass goal_pairs_for_nilsson to get_successors
    successors = get_successors(n, heuristic, goal, goal_pairs=goal_pairs_for_nilsson)
    # go to next n if no successors
    if not successors:
      continue

    for successor in successors:
      existing_node_in_open = None
      existing_node_in_closed = None

      # determine if the node is in OPEN
      for open_node in OPEN:
        if open_node == successor:
          existing_node_in_open = open_node
          break

      # determine if the node si in CLOSE
      if existing_node_in_open is None:
        for closed_node in CLOSED:
          if closed_node == successor:
            existing_node_in_closed = closed_node
            break

      # if successor has lower cost, replace the current node in OPEN
      if existing_node_in_open:
        if successor < existing_node_in_open:
          OPEN.remove(existing_node_in_open)
          OPEN.append(successor)

      # if successor has lower cost, remove from CLOSED then add to OPEN
      elif existing_node_in_closed:
        if successor < existing_node_in_closed:
          CLOSED.remove(existing_node_in_closed)
          OPEN.append(successor)
      # otherwise just add it to OPEN
      else:
        OPEN.append(successor)

  return None

If you intend to run it locally, the `astar_in.txt` may not be available in your device. Run the hidden code cell below to generate it in local directory. Or you may want to experiment on other test cases, just modify the contents of the text file.

In [ ]:
# @title
content = """start
2 1 6
4 * 8
7 5 3
goal
1 2 3
8 * 4
7 6 5"""

with open("astar_in.txt", "w") as file:
    file.write(content)

Running the A* algorithm!!!

In [ ]:
misplaced = a_star('astar_in.txt', heuristic_misplaced);

Using heuristic_misplaced.
Initializing with state: 
['2', '1', '6']
['4', '*', '8']
['7', '5', '3']

f(7) = g(0) + h(7)

Searching for state: 
['1', '2', '3']
['8', '*', '4']
['7', '6', '5']

f(0) = g(0) + h(0)

State has f=7, g=0, h=7.
State has f=9, g=1, h=8.
State has f=9, g=1, h=8.
State has f=9, g=1, h=8.
State has f=9, g=1, h=8.
State has f=9, g=2, h=7.
State has f=10, g=2, h=8.
State has f=10, g=2, h=8.
State has f=10, g=2, h=8.
State has f=10, g=2, h=8.
State has f=10, g=2, h=8.
State has f=10, g=3, h=7.
State has f=10, g=3, h=7.
State has f=10, g=3, h=7.
State has f=10, g=4, h=6.
State has f=10, g=4, h=6.
State has f=10, g=4, h=6.
State has f=11, g=2, h=9.
State has f=11, g=2, h=9.
State has f=11, g=3, h=8.
State has f=11, g=3, h=8.
State has f=11, g=3, h=8.
State has f=11, g=4, h=7.
State has f=11, g=4, h=7.
State has f=11, g=4, h=7.
State has f=11, g=4, h=7.
State has f=11, g=4, h=7.
State has f=12, g=4, h=8.
State has f=12, g=4, h=8.
State has f=12, g=5, h=7.
State has f=1

In [ ]:
manhattan = a_star('astar_in.txt', heuristic_manhattan);

Using heuristic_manhattan.
Initializing with state: 
['2', '1', '6']
['4', '*', '8']
['7', '5', '3']

f(12) = g(0) + h(12)

Searching for state: 
['1', '2', '3']
['8', '*', '4']
['7', '6', '5']

f(0) = g(0) + h(0)

State has f=12, g=0, h=12.
State has f=13, g=1, h=12.
State has f=13, g=1, h=12.
State has f=14, g=2, h=12.
State has f=14, g=2, h=12.
State has f=13, g=3, h=10.
State has f=14, g=4, h=10.
State has f=15, g=1, h=14.
State has f=15, g=1, h=14.
State has f=15, g=3, h=12.
State has f=15, g=5, h=10.
State has f=16, g=2, h=14.
State has f=15, g=3, h=12.
State has f=16, g=2, h=14.
State has f=16, g=4, h=12.
State has f=16, g=2, h=14.
State has f=16, g=2, h=14.
State has f=16, g=4, h=12.
State has f=16, g=4, h=12.
State has f=16, g=4, h=12.
State has f=16, g=4, h=12.
State has f=17, g=5, h=12.
State has f=17, g=5, h=12.
State has f=17, g=3, h=14.
State has f=17, g=5, h=12.
State has f=16, g=6, h=10.
State has f=17, g=3, h=14.
State has f=16, g=4, h=12.
State has f=17, g=3, h=14.
St

In [ ]:
nilsson = a_star('astar_in.txt', heuristic_nilsson);

Using heuristic_nilsson.
Initializing with state: 
['2', '1', '6']
['4', '*', '8']
['7', '5', '3']

f(60) = g(0) + h(60)

Searching for state: 
['1', '2', '3']
['8', '*', '4']
['7', '6', '5']

f(0) = g(0) + h(0)

State has f=60, g=0, h=60, P=12, S=16.
State has f=64, g=1, h=63, P=12, S=17.
State has f=64, g=1, h=63, P=12, S=17.
State has f=65, g=2, h=63, P=12, S=17.
State has f=65, g=2, h=63, P=12, S=17.
State has f=64, g=3, h=61, P=10, S=17.
State has f=62, g=4, h=58, P=10, S=16.
State has f=66, g=1, h=65, P=14, S=17.
State has f=66, g=1, h=65, P=14, S=17.
State has f=66, g=3, h=63, P=12, S=17.
State has f=58, g=4, h=54, P=12, S=14.
State has f=62, g=5, h=57, P=12, S=15.
State has f=62, g=5, h=57, P=12, S=15.
State has f=63, g=6, h=57, P=12, S=15.
State has f=62, g=7, h=55, P=10, S=15.
State has f=46, g=8, h=38, P=8, S=10.
State has f=50, g=9, h=41, P=8, S=11.
State has f=52, g=9, h=43, P=10, S=11.
State has f=53, g=10, h=43, P=10, S=11.
State has f=56, g=9, h=47, P=8, S=13.
State has

## Summary of Results

In [ ]:
import pandas as pd

results = pd.DataFrame([misplaced[1:],
                        manhattan[1:],
                        nilsson[1:]],
                       columns=["Optimal Path Length", "Search Cost", "Nodes Discovered"],
                       index = ["Misplaced", "Manhattan", "Nilsson"])

results

,Optimal Path Length,Search Cost,Nodes Discovered
Misplaced,18,1859,3163
Manhattan,18,241,418
Nilsson,18,32,62


## Analysis and conclusion

1. **A better heuristic function saves memory.**<br>

$\quad$It is evident that the Nilsson's sequence score generated the least amount of nodes, saving memory. A better heuristic saves more memory because the frontier (OPEN) and the searched nodes (CLOSED) stores significantly less number of states. Nilsson's heuristic is only ~2% of the total space used in the misplaced tiles heuristic, and only ~15% of the memory used in Manhattan. It follows that the Manhattan heuristic performs better than the misplaced heuristic in terms of memory efficiency.

2. **A better heuristic function has a lower search cost.**<br>

$\quad$Search cost relates to the time complexity of the algorithm. The higher the search cost, the more computational resources are used or the more time is spent in searching for a solution. The Nilsson sequence score heuristic, again, reigns supreme in this category. It only searched 32 nodes to get to the solution, while Manhattan heuristic searched 241 nodes, and the misplaced heuristic astonishingly searched 1859 nodes. We can also say that the Manhattan heuristic is better than the misplaced heuristic.

3. **A better heuristic function produces higher $h(n)$.**<br>

$\quad$This may not be evident at first, but if we look at the values of the evaluation functions, the better heuristic (Nilsson's sequence score) produces a much higher evaluation value $f$, or accurately, a higher heuristic value $h$ for most time $t$. This hints that a better heuristic function doesn't *overly* underestimate the cost of reaching the goal. Estimation still must follow admissibility and should *never* overestimate the cost of reaching the goal.

4. **Admissible heuristics produce optimal solutions.**<br>

$\quad$We will not delve into proving the admissibility of each heuristic function. However, if we assume that the Nilsson's sequence score is admissible, it is safe to say that the Manhattan and misplaced heuristics are also admissible because $h_\text{Nilsson's}(n) > h_\text{Manhattan}(n) > h_\text{Misplaced}(n)$ based on Conclusion #3. We can see from our solutions that *all* of the heuristics found the **optimal** path to reach the goal state, in this case, all the heuristic functions found a solution that required 18 steps to reach the goal from the initial state. With that, regardless of search cost or memory used, A\* produces an optimal solution if it exists.